# 04 — Is volatility rough? Hurst estimates, real vs sim

H of the realized-vol proxy via DFA, R/S and the GJR smoothness regression. Preregistered: real BTC vol is rough (H well below 0.5); the v1 exp-kernel sim is NOT rough (H ≈ 0.5) — the Jusselin–Rosenbaum prediction our Stage-2 power-law-kernel experiment then attacks.

In [ ]:
FIXTURE_ONLY = True  # papermill parameter

In [ ]:
import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from quantsim_research.config import repo_root

FIXTURE = repo_root() / "data" / "fixtures" / "binance-um" / "trades" / "BTCUSDT-2024-01-07-h00.csv"
ARTIFACTS = repo_root() / "artifacts" / "notebooks"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
df = pl.read_csv(FIXTURE)
ts_ns = df["ts_ns"].to_numpy()
price = df["price_e8"].to_numpy() / 1e8
side = df["side"].to_numpy()
print(f"fixture: {len(df)} trades, {(ts_ns[-1]-ts_ns[0])/1e9:.0f}s span")


In [ ]:
from quantsim_research.hurst.estimators import dfa_hurst, gjr_smoothness_hurst, rescaled_range_hurst
from quantsim_research.hurst.vol_proxy import realized_vol

# Fixture-scale: 10s RV bins over the hour (coarse; the full study uses
# 5-min bins over a month). Small-sample caveats apply to every number here.
rv = realized_vol(ts_ns, price, bin_secs=10.0)
log_rv = np.log(rv[rv > 0])
print(f"{rv.size} RV bins")
print(f"DFA H       = {dfa_hurst(log_rv):.3f}")
print(f"R/S H       = {rescaled_range_hurst(log_rv):.3f}")
print(f"GJR H       = {gjr_smoothness_hurst(log_rv, max_lag=min(50, rv.size // 5)):.3f}")


In [ ]:
# Calibration sanity inline: the estimators recover known H on synthetic fGn.
rng = np.random.default_rng(0)


def fgn(n, h):
    def g(k):
        return 0.5 * (abs(k - 1) ** (2 * h) - 2 * abs(k) ** (2 * h) + abs(k + 1) ** (2 * h))
    gg = np.array([g(k) for k in range(n)])
    r = np.concatenate([gg, gg[-2:0:-1]])
    lam = np.fft.fft(r).real
    lam[lam < 0] = 0
    m_ = r.size
    w = rng.standard_normal(m_) + 1j * rng.standard_normal(m_)
    return np.fft.fft(np.sqrt(lam / (2 * m_)) * w).real[:n]


for h_true in (0.1, 0.5):
    print(f"H={h_true}: DFA -> {dfa_hurst(fgn(4096, h_true)):.3f}")
